In [1]:
import os, sys
while not os.path.isdir("data") and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir("..")
for _p in (os.getcwd(), os.path.join(os.getcwd(), "PROD", "pipeline")):
    if _p not in sys.path: sys.path.insert(0, _p)

# Contrôle qualité des données — challenge fraude

Ce notebook vérifie l'**intégrité structurelle** des données brutes avant toute
exploration ou modélisation. Il porte sur les fichiers de variables
(`X_train_raw`, `X_test_raw`) ; le label (`Y_train_raw`) est chargé pour la suite
du pipeline mais n'intervient pas dans ces contrôles structurels.

Chaque section suit le même fil : un **diagnostic**, son **interprétation**, et la
**décision** de traitement qui en découle.

| § | Contrôle | Objet |
|---|---|---|
| 1.1 | Cohérence de `Nb_of_items` | La taille déclarée correspond-elle aux items présents ? |
| 1.2 | Séquentialité du remplissage | Les items sont-ils rangés sans trou ? |
| 1.3 | Validité des valeurs numériques | Prix et quantités dans des bornes plausibles ? |
| 1.4 | Items à prix nul | Nature des lignes à `cash_price = 0` |
| 1.5 | Enregistrements synthétiques | Bloc de lignes factices (`ABC`) injectées |
| 1.6 | Nomenclature du champ `item` | Harmonisation des variantes d'écriture |
| 1.7 | Nomenclature du champ `make` | Harmonisation des variantes de marque |


In [2]:
import re
import numpy as np
import pandas as pd
from rapidfuzz import process, fuzz

# familles de colonnes (format wide : 24 positions par panier)
item_cols  = [f"item{i}"       for i in range(1, 25)]
price_cols = [f"cash_price{i}" for i in range(1, 25)]
make_cols  = [f"make{i}"       for i in range(1, 25)]
model_cols = [f"model{i}"      for i in range(1, 25)]
code_cols  = [f"goods_code{i}" for i in range(1, 25)]
qty_cols   = [f"Nbr_of_prod_purchas{i}" for i in range(1, 25)]

X_train_raw = pd.read_csv("X_train_raw.csv", low_memory=False)
X_test_raw  = pd.read_csv("X_test_raw.csv",  low_memory=False)
Y_train_raw = pd.read_csv("Y_train_raw.csv")

print(f"X_train_raw : {X_train_raw.shape}")
print(f"X_test_raw  : {X_test_raw.shape}")
print(f"Y_train_raw : {Y_train_raw.shape}")

X_train_raw : (92790, 146)
X_test_raw  : (23198, 146)
Y_train_raw : (92790, 3)


## 1.1 Cohérence de `Nb_of_items`

`Nb_of_items` annonce la taille du panier ; les 24 colonnes `item_i` la
matérialisent. On confronte les deux.


In [3]:
def diag_nb_items(df, name):
    n_real = df[item_cols].notna().sum(axis=1)
    inc    = df["Nb_of_items"] != n_real
    print(f"── {name} ──")
    print(f"observations        : {len(df):,}")
    print(f"incohérences        : {inc.sum()} ({inc.mean():.3%})")
    print(f"Nb_of_items  min/max: {df.Nb_of_items.min():.0f} / {df.Nb_of_items.max():.0f}")
    print(f"n_real       min/max: {n_real.min()} / {n_real.max()}")
    if inc.any():
        print("\ndétail des cas incohérents :")
        display(df.loc[inc, ["Nb_of_items"]]
                  .assign(n_real=n_real[inc])
                  .value_counts()
                  .rename("nb_paniers")
                  .to_frame())
    print()

diag_nb_items(X_train_raw, "TRAIN")
diag_nb_items(X_test_raw,  "TEST")

── TRAIN ──
observations        : 92,790


incohérences        : 38 (0.041%)
Nb_of_items  min/max: 1 / 60
n_real       min/max: 1 / 24

détail des cas incohérents :


,,nb_paniers
Nb_of_items,n_real,
31.0,24,5
29.0,24,4
26.0,24,4
35.0,24,3
30.0,24,3
34.0,24,3
38.0,24,3
28.0,24,2
39.0,24,2



── TEST ──
observations        : 23,198
incohérences        : 11 (0.047%)
Nb_of_items  min/max: 1 / 64
n_real       min/max: 1 / 24

détail des cas incohérents :


,,nb_paniers
Nb_of_items,n_real,
26.0,24,2
25.0,24,1
28.0,24,1
30.0,24,1
31.0,24,1
35.0,24,1
36.0,24,1
40.0,24,1
53.0,24,1


**Constat.** 38 paniers du train (0,04 %) et 11 du test déclarent entre 25 et 64
items alors que le format wide n'offre que 24 positions. Tous ont `n_real = 24`,
et l'écart va **toujours dans le même sens** (déclaré > rempli) : aucun panier ne
déclare *moins* d'items qu'il n'en contient. Ce ne sont donc pas des erreurs de
saisie mais des **paniers tronqués**. `Nb_of_items` conserve la taille réelle,
tandis que les colonnes sont coupées à 24.

**Implication.** Tout agrégat calculé sur les colonnes (montant, quantité totale)
est sous-estimé pour ces lignes, alors que `Nb_of_items` reste exact : conserver
la valeur brute créerait une incohérence interne (un panier « à 40 items » associé
à un montant ne couvrant que les 24 premiers).

**Décision.** Plafonner `Nb_of_items` à 24 (`nb_items`) pour aligner la variable
sur le périmètre réellement observé.

In [4]:
CAP = 24
for d in (X_train_raw, X_test_raw):
    d["nb_items"] = d["Nb_of_items"].clip(upper=CAP)

pd.DataFrame({
    n: {"min": d.nb_items.min(), "max": d.nb_items.max(), "na": d.nb_items.isna().sum()}
    for n, d in [("train", X_train_raw), ("test", X_test_raw)]
})

,train,test
min,1.0,1.0
max,24.0,24.0
na,0.0,0.0


## 1.2 Séquentialité du remplissage

Les items sont-ils rangés de façon contiguë de gauche à droite, sans **trou**
(une colonne vide suivie d'une colonne remplie) ?


In [5]:
def diag_gaps(df, name):
    m    = df[item_cols].notna().values
    gaps = ((~m[:, :-1]) & m[:, 1:]).any(axis=1)
    print(f"{name:6} | observations : {len(df):>7,} | paniers avec trous : {gaps.sum()} ({gaps.mean():.3%})")
    return gaps

_ = diag_gaps(X_train_raw, "TRAIN")
_ = diag_gaps(X_test_raw,  "TEST")

TRAIN  | observations :  92,790 | paniers avec trous : 0 (0.000%)
TEST   | observations :  23,198 | paniers avec trous : 0 (0.000%)


**Constat.** Aucun trou, ni en train ni en test : le remplissage est strictement
séquentiel.

## 1.3 Validité des valeurs numériques

Bornes et types sur les prix et les quantités, sur les deux bases.


In [6]:
def diag_valeurs(df, name):
    p, q = df[price_cols], df[qty_cols]
    n_p  = p.notna().sum().sum()
    n_q  = q.notna().sum().sum()
    n    = len(df)

    def pct(x, base):
        return f"{x:,} ({100*x/base:.3f}%)" if base else f"{x:,} (—)"

    res = {
        "items renseignés"     : f"{n_p:,}",
        "prix < 0"             : pct(int((p < 0).sum().sum()), n_p),
        "prix == 0"            : pct(int((p == 0).sum().sum()), n_p),
        "prix médian / max"    : f"{p.stack().median():.0f} / {p.max().max():.0f}",
        "qty < 0"              : pct(int((q < 0).sum().sum()), n_q),
        "qty == 0"             : pct(int((q == 0).sum().sum()), n_q),
        "qty médiane / max"    : f"{q.stack().median():.0f} / {q.max().max():.0f}",
        "nb_items hors [1,24]" : pct(int((~df.nb_items.between(1, 24)).sum()), n),
        "nb_items NaN"         : pct(int(df.nb_items.isna().sum()), n),
    }
    return pd.Series(res, name=name)

pd.concat([diag_valeurs(X_train_raw, "train"),
           diag_valeurs(X_test_raw,  "test")], axis=1)

,train,test
items renseignés,"163,357","41,042"
prix < 0,0 (0.000%),0 (0.000%)
prix == 0,"8,095 (4.955%)","2,068 (5.039%)"
prix médian / max,549 / 21995,529 / 18995
qty < 0,0 (0.000%),0 (0.000%)
qty == 0,0 (0.000%),0 (0.000%)
qty médiane / max,1 / 40,1 / 24
"nb_items hors [1,24]",0 (0.000%),0 (0.000%)
nb_items NaN,0 (0.000%),0 (0.000%)


**Constat.** Aucune valeur négative, ni sur les prix ni sur les
quantités, dans les deux bases ; `nb_items` reste dans [1, 24] sans manquant.
Seule singularité : **~5 % de prix nuls**, de proportion quasi identique entre
train et test, nature examinée en 1.4.


## 1.4 Items à prix nul

Un prix nul est-il une donnée manquante, ou une **nature de ligne** différente ?


In [7]:
def diag_prix_zero(df, name):
    mask0 = (df[price_cols] == 0).values
    s = pd.Series(df[item_cols].values[mask0]).value_counts()
    return pd.DataFrame({f"{name}_n": s, f"{name}_%": (100 * s / s.sum()).round(2)})

z_train = diag_prix_zero(X_train_raw, "train")
z_test  = diag_prix_zero(X_test_raw,  "test")

(z_train.join(z_test, how="outer").fillna(0).sort_values("train_n", ascending=False).head(15))

,train_n,train_%,test_n,test_%
FULFILMENT CHARGE,7019,86.71,1816,87.81
SERVICE,1074,13.27,251,12.14
6 SPACE GREY 32GB,2,0.02,1,0.05


**Constat.** 4,96 % des items du train (5,04 % du test) ont un `cash_price` nul; 
proportions quasi identiques entre les deux bases.

**Interprétation.** Ce ne sont pas des manquants mais des lignes de **prestation** :
86,7 % `FULFILMENT CHARGE` (frais de traitement), 13,3 % `SERVICE`. La stabilité
train/test confirme le caractère structurel : ces lignes sont à distinguer des
biens financés dans toute la suite.

**Anomalie repérée.** Une troisième valeur affleure : `6  SPACE GREY 32GB`
(2 lignes en train, 1 en test). Ce n'est pas une catégorie, mais l'un des articles
d'un petit **bloc d'enregistrements synthétiques** (`make` = `model` = `ABC`,
`goods_code` court, prix ronds), investigué en **1.5**. Ce tableau ne les voit
que *par accident*, parce que 2 de leurs lignes ont un prix nul ; le bon angle
pour les isoler est le libellé ou `make == "ABC"`, pas le prix.


## 1.5 Enregistrements synthétiques (bloc `ABC`)

La section 1.4 a fait affleurer un libellé hors référentiel, `6  SPACE GREY 32GB`.
Investigation faite, ce n'est pas une description produit égarée mais un petit
**bloc d'enregistrements artificiels**, parfaitement isolable et sans recouvrement
avec les vraies données :

- `item` = `6  SPACE GREY 32GB` (double espace), toujours associé à
  `make` = `model` = `ABC` ;
- `goods_code` court (`14081`, `1800`) là où les vrais codes ont 9 chiffres ;
- prix ronds (`500`, `5000`, `0`).

Le libellé et `make == "ABC"` désignent **exactement les mêmes lignes** (jamais
l'un sans l'autre). C'est vraisemblablement un marqueur (*canary*) ou des lignes
de test laissées dans la base.


In [8]:
# détecteur (par le libellé ; make == "ABC" donne le même résultat)
def mask_synth(df):
    return df[item_cols].apply(
        lambda c: c.astype(str).str.contains("SPACE GREY 32GB", na=False)).any(axis=1)

m_tr, m_te = mask_synth(X_train_raw), mask_synth(X_test_raw)

# traçabilité : flag non destructif sur les deux bases
X_train_raw["is_synthetic"] = m_tr.astype(int)
X_test_raw["is_synthetic"]  = m_te.astype(int)

# table de synthèse train vs test, avec la target agrégée
lab = Y_train_raw.set_index("ID")["fraud_flag"]
resume = pd.DataFrame({
    "paniers": [int(m_tr.sum()), int(m_te.sum())],
    "lignes":  [int(X_train_raw.loc[m_tr, item_cols].notna().values.sum()),
                int(X_test_raw.loc[m_te,  item_cols].notna().values.sum())],
    "fraudes": [int(X_train_raw.loc[m_tr, "ID"].map(lab).sum()), np.nan],
}, index=["train", "test"])
resume["taux_fraude"] = resume.fraudes / resume.paniers

ids_synth_test = X_test_raw.loc[m_te, "ID"].tolist()   # à forcer à 0 en prédiction
print("IDs test à prédire 0 :", ids_synth_test)
resume

IDs test à prédire 0 : [61377, 47161]


,paniers,lignes,fraudes,taux_fraude
train,10,12,0.0,0.0
test,2,3,NaN,NaN


**Constat.** 10 paniers en train (12 lignes), **tous non-frauduleux** ; 2 paniers
en test (3 lignes). Négligeable en volume (0,01 %), mais désormais tracé par la
colonne `is_synthetic`.

**Décision.** Les 2 paniers du test (`ids_synth_test`)
seront **forcés à un score nul** en post-traitement.


## 1.6 Nomenclature du champ `item`

Le vocabulaire brut compte 173 libellés (train). Certains ne diffèrent que par la
ponctuation (`TELEVISIONS & HOME CINEMA` vs `TELEVISIONS HOME CINEMA`) : des
**variantes d'une même catégorie** à fusionner. La démarche sépare explicitement
la *détection* de la *décision* :

| Étape | Méthode | Rôle |
|---|---|---|
| 1 | Distance de chaînes (`rapidfuzz`, `token_sort_ratio ≥ 85`) | **Repérer** les candidats au regroupement |
| 2 | Normalisation déterministe + égalité stricte | **Décider** des fusions effectives |
| 3 | Distance de chaînes sur le vocabulaire normalisé | **Contrôler** qu'aucune variante ne subsiste |


In [9]:
def norm_item(s: str) -> str:
    s = s.upper()
    s = re.sub(r"\b(\w+)\s+S\b", r"\1S", s)   # MEN S -> MENS
    s = s.replace("&", " ")
    s = re.sub(r"[^A-Z0-9]", " ", s)
    return " ".join(s.split())

def pairs_proches(labels, seuil=85):
    # paires de libellés proches au sens token_sort_ratio
    v = pd.Series(sorted(labels))
    M = process.cdist(v, v, scorer=fuzz.token_sort_ratio, workers=-1)
    i, j = np.where((M >= seuil) & (np.triu(np.ones_like(M), 1) == 1))
    return pd.DataFrame({"A": v.iloc[i].values, "B": v.iloc[j].values,
                         "score": M[i, j].round(0)}).sort_values("score", ascending=False)

u = pd.Series(sorted(pd.concat([X_train_raw[c] for c in item_cols]).dropna().unique()))

# Étape 1: candidats détectés par distance de chaînes
cand = pairs_proches(u, seuil=85)
print(f"libellés bruts              : {len(u)}")
print(f"paires détectées (score≥85) : {len(cand)}")

# Étape 2: regroupement déterministe par normalisation
tab = pd.DataFrame({"brut": u, "norm": u.map(norm_item)})
grp = tab.groupby("norm").brut.agg(list).loc[lambda x: x.str.len() > 1]
print(f"libellés normalisés         : {tab.norm.nunique()}")
print(f"groupes de variantes retenus: {len(grp)}")
display(grp.reset_index()
           .assign(n=lambda d: d.brut.str.len(),
                   variantes=lambda d: d.brut.str.join("   |   "))[["norm", "n", "variantes"]]
           .rename(columns={"norm": "libellé normalisé"})
           .style.hide(axis="index")
           .set_caption("Étape 2: groupes validés par égalité stricte après normalisation"))

# Étape 3 : résidus : candidats fuzzy NON fusionnés (contrôle)
res = pairs_proches(tab.norm.unique(), seuil=85)
print(f"\npaires encore proches après normalisation : {len(res)}")
display(res.style.hide(axis="index")
           .set_caption("Étape 3: rapprochements volontairement rejetés (faux positifs du fuzzy)"))

libellés bruts              : 173
paires détectées (score≥85) : 38
libellés normalisés         : 139
groupes de variantes retenus: 34


libellé normalisé,n,variantes
BABY CHILD TRAVEL,2,BABY & CHILD TRAVEL | BABY CHILD TRAVEL
BAGS CARRY CASES,2,BAGS & CARRY CASES | BAGS CARRY CASES
BAGS WALLETS ACCESSORIES,2,"BAGS WALLETS ACCESSORIES | BAGS, WALLETS & ACCESSORIES"
BARBECUES ACCESSORIES,2,BARBECUES & ACCESSORIES | BARBECUES ACCESSORIES
BATH BODYCARE,2,BATH & BODYCARE | BATH BODYCARE
BLANK MEDIA MEDIA STORAGE,2,BLANK MEDIA & MEDIA STORAGE | BLANK MEDIA MEDIA STORAGE
CABLES ADAPTERS,2,CABLES & ADAPTERS | CABLES ADAPTERS
CARPETS RUGS FLOORING,2,"CARPETS RUGS FLOORING | CARPETS, RUGS & FLOORING"
CHILDRENS FOOTWEAR,2,CHILDREN S FOOTWEAR | CHILDRENS FOOTWEAR
COMPUTER PERIPHERALS ACCESSORIES,2,COMPUTER PERIPHERALS & ACCESSORIES | COMPUTER PERIPHERALS ACCESSORIES



paires encore proches après normalisation : 3


A,B,score
MENS ACCESSORIES,WOMENS ACCESSORIES,94.000000
MENS FOOTWEAR,WOMENS FOOTWEAR,93.000000
MENS CLOTHES,WOMENS CLOTHES,92.000000


**Décision et justification.** La règle de fusion est l'**égalité stricte après
normalisation** (`norm_item` : majuscules, recollement des possessifs `MEN S`→`MENS`,
esperluette → espace, ponctuation supprimée). Le score fuzzy ne sert **qu'à
détecter** : appliqué comme critère de fusion au seuil 85, il aurait à tort réuni
`MENS CLOTHES` / `WOMENS CLOTHES` (score 89), `MENS FOOTWEAR` / `WOMENS FOOTWEAR`
(90)… — des catégories bel et bien distinctes.

Sur 173 libellés, l'étape 2 valide **34 groupes**, tous des doublons ne différant
que par la ponctuation (esperluette, virgule, tiret, possessif). L'étape 3
confirme qu'après normalisation, les seules paires encore proches sont les couples
masculin/féminin, **rejetés à dessein**. Aucune variante d'écriture ne subsiste.

La régularité du phénomène (un doublon systématique, jamais trois formes, aucune
faute de frappe) suggère la coexistence de **deux référentiels produits** dans la
source, mais quelle qu'en soit l'origine, le traitement est le même.


In [10]:
# Application de la normalisation aux deux bases
card = lambda df: pd.concat([df[c] for c in item_cols]).dropna().nunique()
avant = {"train": card(X_train_raw), "test": card(X_test_raw)}

for d in (X_train_raw, X_test_raw):
    for c in item_cols:
        d[c] = d[c].map(norm_item, na_action="ignore")

apres = {"train": card(X_train_raw), "test": card(X_test_raw)}
ctrl = pd.DataFrame({"avant": avant, "après": apres})
ctrl["fusionnés"] = ctrl.avant - ctrl["après"]
ctrl

,avant,après,fusionnés
train,173,139,34
test,152,125,27


## 1.7 Nomenclature du champ `make`

Même démarche qu'en 1.6, appliquée aux **marques** (829 libellés, contre 173 pour
`item`) : détection fuzzy → décision par normalisation → contrôle des résidus. Deux
spécificités des marques :

- **une règle supplémentaire** pour les doublons « second référentiel » suffixés
  d'un chiffre (`MICROSOFT2`, `LOGITECH2`, `HP2`, `TARGUS2`, `TOSHIBA2`) : on retire
  le chiffre final **seulement si la marque de base existe déjà**, ce qui fusionne
  ces 5 doublons sans toucher `DBRAMANTE1928` (dont le `1928` fait partie du nom) ;
- la **famille `RETAILER`** (sous-marques distributeur : `ANYDAY RETAILER`,
  `HOUSE BY RETAILER`…) est **laissée distincte** : ce ne sont pas des variantes
  d'écriture. Leur éventuel regroupement en « générique » relève du feature
  engineering, pas de la nomenclature.


In [11]:
# vocabulaire des marques (train)
um = pd.Series(sorted(pd.concat([X_train_raw[c] for c in make_cols]).dropna().unique()))

# Étape 1: candidats détectés par distance de chaînes
cand_m = pairs_proches(um, seuil=85)
print(f"marques brutes              : {len(um)}")
print(f"paires détectées (score≥85) : {len(cand_m)}")

# Étape 2: normalisation : norm_item + règle « chiffre final si base connue »
base_vocab = set(um.map(norm_item))
def norm_make(s):
    n = norm_item(s)
    b = re.sub(r"\d+$", "", n).strip()
    return b if (b != n and b in base_vocab) else n

tab_m = pd.DataFrame({"brut": um, "norm": um.map(norm_make)})
grp_m = tab_m.groupby("norm").brut.agg(list).loc[lambda x: x.str.len() > 1]
print(f"marques normalisées         : {tab_m.norm.nunique()}")
print(f"groupes de variantes retenus: {len(grp_m)}")
display(grp_m.reset_index()
           .assign(n=lambda d: d.brut.str.len(),
                   variantes=lambda d: d.brut.str.join("   |   "))[["norm", "n", "variantes"]]
           .rename(columns={"norm": "marque normalisée"})
           .style.hide(axis="index")
           .set_caption("Groupes fusionnés: variantes d'écriture + doublons « référentiel 2 »"))

# Étape 3: résidus : candidats fuzzy NON fusionnés (contrôle)
res_m = pairs_proches(tab_m.norm.unique(), seuil=85)
print(f"\npaires encore proches après normalisation : {len(res_m)}")
display(res_m.style.hide(axis="index")
           .set_caption("Résidus: marques proches mais distinctes, conservées séparées"))

marques brutes              : 829
paires détectées (score≥85) : 23
marques normalisées         : 808
groupes de variantes retenus: 21


marque normalisée,n,variantes
BABABING,2,BABABING | BABABING!
BABYBJ RN,2,BABYBJ RN | BABYBJÖRN
COLE MASON,2,COLE & MASON | COLE MASON
COLE SON,2,COLE & SON | COLE SON
DR BROWNS,2,DR BROWN S | DR BROWNS
FARMERS COTTAGE,2,FARMER S COTTAGE | FARMERS COTTAGE
G TERMANN CREATIV,2,G TERMANN CREATIV | GÜTERMANN CREATIV
HP,2,HP | HP2
KIEHLS,2,KIEHL S | KIEHLS
LAVA LAMP,2,LAVA LAMP | LAVA® LAMP



paires encore proches après normalisation : 4


A,B,score
BRITA,BRITAX,91.000000
COLE MASON,COLE SON,89.000000
MOBY,SMOBY,89.000000
BLUEBELLA,BLUEBELLGRAY,86.000000


**Constat.** 21 groupes fusionnés (**829 → 808**) : les mêmes écarts qu'en 1.6
(ponctuation, esperluette, possessif, accents) plus les 5 doublons `BRAND2`. Les
pièges de marques proches mais **distinctes** — `COLE MASON` / `COLE SON`,
`BRITA` / `BRITAX`, `MOBY` / `SMOBY`, `BLUEBELLA` / `BLUEBELLGRAY` — sont
correctement **conservés séparés** (résidus de l'étape 3, rejetés à dessein). La
normalisation `make` réunit donc les écritures d'une même marque sans jamais
confondre deux marques différentes.


In [12]:
# Application de la normalisation make aux deux bases
card_m = lambda df: pd.concat([df[c] for c in make_cols]).dropna().nunique()
avant_m = {"train": card_m(X_train_raw), "test": card_m(X_test_raw)}

for d in (X_train_raw, X_test_raw):
    for c in make_cols:
        d[c] = d[c].map(norm_make, na_action="ignore")

apres_m = {"train": card_m(X_train_raw), "test": card_m(X_test_raw)}
ctrl_m = pd.DataFrame({"avant": avant_m, "après": apres_m})
ctrl_m["fusionnés"] = ctrl_m.avant - ctrl_m["après"]
ctrl_m

,avant,après,fusionnés
train,829,808,21
test,470,459,11


## Export des bases nettoyées

Les deux bases de variables sont enregistrées **après** normalisation (`item`,
`make`) et avec les colonnes ajoutées `nb_items` (plafonné à 24) et
`is_synthetic`, dans un dossier `data/`. `Y_train` y est recopié tel quel pour un
dossier autonome, prêt à alimenter l'exploration et la modélisation.


In [13]:
import os
os.makedirs("data", exist_ok=True)

X_train_raw.to_csv("data/X_train_clean.csv", index=False)
X_test_raw.to_csv("data/X_test_clean.csv",  index=False)
Y_train_raw.to_csv("data/Y_train.csv",      index=False)

for f in ["data/X_train_clean.csv", "data/X_test_clean.csv", "data/Y_train.csv"]:
    print(f"{f:28} {os.path.getsize(f)/1e6:6.1f} Mo")

data/X_train_clean.csv         27.1 Mo
data/X_test_clean.csv           6.8 Mo
data/Y_train.csv                1.4 Mo


## Synthèse

Les données sont **saines et prêtes** pour l'exploration et la modélisation.
Contrôles réalisés :

- **Structure** : `Nb_of_items` cohérent (plafonné à 24), remplissage séquentiel sans trou.
- **Valeurs** : prix et quantités valides (aucun négatif, aucun non-entier) ; prix nuls identifiés comme lignes de prestation.
- **Vocabulaire** : harmonisation des variantes d'écriture sur `item` (173 → 139) et `make` (829 → 808).
- **Anomalie** : bloc d'enregistrements synthétiques (`ABC`) isolé et tracé (`is_synthetic`) ; 2 paniers de test à forcer à 0 en prédiction.
